[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/28-generating-audio.ipynb)

# Module 28: Generating Audio: Codecs and TTS

Companion notebook to [Module 28](https://transformers-and-math.drmhse.com/modules/28-generating-audio) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)

## Part 1 — From a waveform to a spectrogram (section 28.1)

In [ ]:
sr = 16_000
t = torch.arange(sr) / sr
wave = torch.where(t < 0.5, torch.sin(2 * math.pi * 300 * t), torch.sin(2 * math.pi * 1200 * t))
wave[int(0.75 * sr)] += 5                                  # a click
spec = torch.stft(wave, n_fft=400, hop_length=160, window=torch.hann_window(400), return_complex=True).abs()
fig, ax = plt.subplots(2, 1, figsize=(10, 5))
ax[0].plot(t, wave); ax[0].set_title('waveform: 16,000 samples')
ax[1].imshow((spec + 1e-3).log(), origin='lower', aspect='auto', extent=[0, 1, 0, sr / 2]); ax[1].set_title(f'spectrogram: {spec.shape[1]} frames × {spec.shape[0]} frequencies')
plt.tight_layout(); plt.show()

## Part 2 — An autoencoder (section 28.2)
Compress 64-number "frames" to an 8-number latent and back.

In [ ]:
basis = torch.randn(8, 64)
frames = lambda n: torch.randn(n, 8) @ basis + 0.05 * torch.randn(n, 64)    # data that really is 8-dimensional
enc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 8))
dec = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 64))
opt = torch.optim.Adam([*enc.parameters(), *dec.parameters()], lr=3e-3)
for step in range(1500):
    x = frames(256)
    loss = F.mse_loss(dec(enc(x)), x)
    opt.zero_grad(); loss.backward(); opt.step()
x = frames(1000)
print(f'reconstruction error {F.mse_loss(dec(enc(x)), x).item():.4f}  vs signal variance {x.var().item():.2f}')

## Part 3 — Residual vector quantization (section 28.3)
Learn several small codebooks one after another, each on what the previous ones left.

In [ ]:
def kmeans(x, k, iters=20):
    c = x[torch.randperm(len(x))[:k]].clone()
    for _ in range(iters):
        assign = torch.cdist(x, c).argmin(1)
        for j in range(k):
            if (assign == j).any():
                c[j] = x[assign == j].mean(0)
    return c

with torch.no_grad():
    z = enc(frames(4000))                                  # latents to quantize
    residual, recon = z.clone(), torch.zeros_like(z)
    print(f'stage 0: error {z.pow(2).mean():.4f}')
    for stage in range(1, 7):
        book = kmeans(residual, 16)
        q = book[torch.cdist(residual, book).argmin(1)]
        recon += q; residual -= q
        print(f'stage {stage}: {4 * stage:2} bits per frame, error {residual.pow(2).mean():.4f}')
    audio_err = F.mse_loss(dec(recon), dec(z)).item()
print(f'decoding the 6-codebook reconstruction differs from decoding the exact latent by {audio_err:.4f}')

## Your turn
1. How many codes per second does a 12 Hz codec with 16 codebooks produce? And how many raw samples is a second of 24 kHz audio?
2. Compare one 256-entry codebook (8 bits per frame) with two 16-entry residual stages (also 8 bits). Then note what one codebook matching six 16-entry stages would need: $16^6 \approx 16.8$ million entries.